# NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV

Extract Debezium CDC từ `lh_vv_bronze.dbo.partner_raw_data` vào `lh_vv_silver.dbo.slv_*`.
Bảng, cột, khoá, thứ tự chạy đọc từ bảng control `lh_vv_bronze.ctrl`; hàm dùng chung lấy từ `NB_LIB_EXTRACT_RAWDATA` (`%run`).

| | |
|---|---|
| Gọi bởi | `PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00` → ForEach tuần tự → If_HasWork. Nhánh có việc gọi notebook này với `partner_raw_data` gắn cứng. Không có Switch |
| Đọc | `ctrl_mng_pipeline_config`, `ctrl_cfg_schema_registry`, `ctrl_mng_watermark`, `ctrl_cdc_state`, raw: file mới theo `_delta_log` (VERSION) hoặc `VERSION AS OF` toàn bộ (FULL) |
| Ghi | `slv_*` (MERGE, xoá mềm), `ctrl_cdc_state`, `ctrl_cdc_reject`, `ctrl_log_run`, `ctrl_log_table_run`, `ctrl_mng_watermark` |
| Chạy song song | Bảng cùng `priority` chạy cùng lúc (`max_parallel` luồng), wave sau chạy khi wave trước xong |
| Chạy lại | An toàn: state theo entity bỏ event đã áp dụng (state không bao giờ lùi); MERGE theo khoá; watermark chỉ tiến khi mọi bảng thành công và không bao giờ lùi |
| Chạy chồng | Khoá nguyên tử trên dòng watermark (`lock_exec_id`, `lock_at`): run thứ 2 → `SKIPPED_CONCURRENT` |

Luồng: nạp + kiểm tra cấu hình → nhận khoá, mở run log → chốt version raw → đọc phần raw mới 1 lần
(VERSION: chỉ file của các commit sau `last_src_version`), phân loại VALID / IGNORED / REJECTED
→ chạy từng wave: parse → lọc event cũ theo state → gộp TOAST → MERGE → commit state, reject, log, watermark → exit JSON.
Mô tả chi tiết: `claude/NB_EXTRACT_PARTNER_CDC_FLOW.md`.

**[03/10 → LIB] Thay đổi 03/10 khi thêm nguồn 3rd-party** — chỉ dời code, **logic không đổi** (đã chạy lại bộ test partner):

| Chỗ trong notebook này | Trước | Sau | Ảnh hưởng |
|---|---|---|---|
| Hằng số | `STATE_COLUMNS` | Lib §13 | Không |
| Mục B | `event_order`, `load_state`, `state_rows` | Lib §13 (`state_rows` thêm `coalesce(is_deleted, false)`; partner luôn có op → như cũ) | Không |
| Mục B | Thân `process_table` (retry, log) | Gọi `process_with_retry` (lib §14) | Không |
| Mục C | Thân `run_waves` | Gọi `run_tables` (lib §14) | Không |
| Mục C | `state_order_sql`, `STATE_NOT_OLDER`, `commit_state` | Lib §13 | Không |
| Mục C | MERGE trong `commit_rejects` | Gọi `merge_rejects` (lib §13) | Không |
| Mục C | `commit_state(results)` / MERGE reject | Truyền thêm `ctx.src_tbl`: điều kiện partition `t.src_tbl = 'partner_raw_data'` (2 bảng partition theo `src_tbl`) | Không đổi kết quả; hết xung đột ghi với luồng 3rd-party |
| Main | Khoá / run log / watermark / nhả khoá trong `run_extract` | Gọi `run_controlled` (lib §14); phần riêng còn lại là `process_run` | Không |
| Main | `load_table_specs(ctx)` | Như cũ; lib mặc định `allowed_modes=("CDC",)`: dòng `pipeline_config` của partner có `load_mode` khác `CDC` → `ConfigError` | Chỉ khi cấu hình sai |

**Deploy:** cập nhật `NB_LIB_EXTRACT_RAWDATA` **trước** notebook này (notebook này gọi hàm mới của lib).
Lib mới + notebook partner cũ vẫn chạy đúng (hàm định nghĩa lại trong notebook cũ ghi đè bản ở lib, cùng logic).


In [1]:
# =============================================================================
# THAM SỐ (toggle "parameters"; pipeline truyền đè)
# =============================================================================
pl_name                 = "PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00"   # @pipeline().Pipeline
run_id                  = ""        # @pipeline().RunId; để trống khi chạy tay
src_schema              = "lh_vv_bronze.dbo"
src_tbl                 = "partner_raw_data"
overlap_minutes         = 10        # không còn dùng (02/10 bỏ kiểu đọc TIME); giữ để pipeline truyền cũ không lỗi
allow_full_scan         = True     # True: cho phép đọc FULL khi không đọc theo version được (lần đầu, sau reset, thiếu log, raw bị tạo lại)
max_parallel            = 12         # số bảng chạy đồng thời trong 1 wave
max_retries             = 1         # thử lại 1 bảng khi gặp lỗi tạm thời (xung đột ghi Delta...)
table_filter            = ""        # vd "slv_products,slv_orders"; rỗng = tất cả bảng active
full_reload             = False     # True: đọc toàn bộ raw, bỏ qua state (nên dùng kèm table_filter)
dry_run                 = False     # True: chỉ đọc, parse, đếm; không ghi bất kỳ bảng nào
stop_on_failure         = False     # True: wave có bảng lỗi thì không chạy các wave sau
cast_null_policy        = "FAIL"    # FAIL: bảng có giá trị thành NULL khi chuyển kiểu -> FAILED trước MERGE, giữ watermark ; WARN: chỉ cảnh báo
running_timeout_minutes = 780       # [SỬA 08/10] 13 giờ, hơn 12 giờ. Khoá dài hơn timeout activity 59 phút

StatementMeta(, 58ce84c9-a336-4b3c-843e-6dba318d52af, 3, Finished, Available, Finished, False)

In [2]:
%run NB_LIB_EXTRACT_RAWDATA

StatementMeta(, 58ce84c9-a336-4b3c-843e-6dba318d52af, 17, Finished, Available, Finished, True)

## Hằng số Debezium

| Hằng số | Ý nghĩa |
|---|---|
| `VALID_OPS`, `UPSERT_OPS` | Op hợp lệ `c r u d`; ghi/sửa là `c r u` (create, read = snapshot, update) |
| `TOAST_SENTINEL` | Debezium gửi chuỗi này cho cột TOAST không đổi (giá trị lớn PostgreSQL không đọc lại) |
| `IGNORE_REASONS` | Lý do event được bỏ qua (chỉ đếm, không phải lỗi) |
| `SOURCE_FIELDS` | Trường cần lấy trong `source` của Debezium |
| `ORDER_ASC`, `ORDER_DESC` | Thứ tự event cũ → mới / mới → cũ; cùng thứ tự thì event xoá được coi là mới hơn (giống notebook cũ) |

In [3]:
# =============================================================================
# HẰNG SỐ DEBEZIUM + CONTEXT NOTEBOOK
# =============================================================================
import uuid                                  # [03/10 → LIB] bỏ import traceback (run_extract dùng run_controlled của lib)
from functools import reduce

from pyspark import StorageLevel
from pyspark.sql.window import Window

VALID_OPS      = ("c", "r", "u", "d")
UPSERT_OPS     = ("c", "r", "u")
TOAST_SENTINEL = "__debezium_unavailable_value"
IGNORE_REASONS = ("TOMBSTONE", "OP_TRUNCATE", "OP_MESSAGE", "UNCONFIGURED", "OUT_OF_SCOPE")
SOURCE_FIELDS  = ("schema", "table", "db", "ts_ms", "lsn")

ORDER_ASC = [F.col("_cdc_ts_ms").asc(), F.col("_cdc_lsn").asc_nulls_first(),
             F.col("_raw_ingested_at").asc(), F.col("_is_delete").asc()]
ORDER_DESC = [F.col("_cdc_ts_ms").desc(), F.col("_cdc_lsn").desc_nulls_last(),
              F.col("_raw_ingested_at").desc(), F.col("_is_delete").desc()]

# [03/10 → LIB] STATE_COLUMNS chuyển sang NB_LIB_EXTRACT_RAWDATA §13 (dùng chung với nguồn 3rd-party)

try:
    NB_NAME = notebookutils.runtime.context.get("currentNotebookName") or "NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV"
except Exception:
    NB_NAME = "NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV"

StatementMeta(, 58ce84c9-a336-4b3c-843e-6dba318d52af, 18, Finished, Available, Finished, False)

## A. Đọc và phân loại raw

| Hàm | Mục đích |
|---|---|
| `objects_cond(objects)` | Điều kiện event thuộc tập (schema, bảng Debezium) |
| `read_raw(ctx)` | Đọc phần raw mới bằng `read_incremental` của thư viện: VERSION (file của commit mới) / FULL |
| `classify_events(raw, scope, configured)` | Parse `source` 1 lần, gắn `_status` VALID / IGNORED / REJECTED + `_reason` |
| `profile_events(classified)` | 1 job đếm theo trạng thái/lý do/bảng nguồn + `watermark_to` (đồng thời nạp cache) |
| `reject_rows(df, ctx, reason, detail)` | Chuẩn hoá event lỗi theo schema `ctrl_cdc_reject` |

In [4]:
# =============================================================================
# A. ĐỌC VÀ PHÂN LOẠI RAW
# =============================================================================
def objects_cond(objects) -> Column:
    """Event thuộc 1 trong các (schema, bảng); schema None = khớp mọi schema.
    Vd objects_cond({("public", "products")}) -> concat_ws('.', _src_object_schema, _src_object) IN ('public.products')"""
    with_schema = sorted(f"{s}.{o}" for s, o in objects if s is not None)
    no_schema = sorted(o for s, o in objects if s is None)
    cond = F.lit(False)
    if with_schema:
        cond = cond | F.concat_ws(".", "_src_object_schema", "_src_object").isin(with_schema)
    if no_schema:
        cond = cond | F.col("_src_object").isin(no_schema)
    return cond


def read_raw(ctx: RunContext) -> DataFrame:
    """4 cột Debezium + thời điểm ingest (_raw_ingested_at) của phần raw cần xử lý.
    Mặc định VERSION: chỉ đọc file được thêm trong các commit (last_src_version, version đã chốt] — không quét cả bảng.
    Cách đọc thực tế + lý do ghi vào ctx.read_mode / ctx.read_note."""
    return read_incremental(ctx, ["before", "after", "source", "op"], "_raw_ingested_at")


def classify_events(raw: DataFrame, scope_objects, configured_objects) -> DataFrame:
    """Gắn _status: VALID | IGNORED (chỉ đếm) | REJECTED (ghi ctrl_cdc_reject), _reason, _reject_detail.
    Luật áp theo thứ tự, luật khớp đầu tiên quyết định. Đổi tên cột payload sang tiền tố "_" để không trùng cột nghiệp vụ."""
    df = raw.withColumn("_s", parse_json_strings(F.col("source"), SOURCE_FIELDS)).select(
        F.col("before").alias("_before"), F.col("after").alias("_after"),
        F.col("source").alias("_source"), F.col("op").alias("_op"), "_raw_ingested_at",
        F.col("_s.schema").alias("_src_object_schema"),
        F.col("_s.table").alias("_src_object"),
        F.col("_s.db").alias("_src_db"),
        F.expr("TRY_CAST(_s.ts_ms AS BIGINT)").alias("_cdc_ts_ms"),
        F.expr("TRY_CAST(_s.lsn AS BIGINT)").alias("_cdc_lsn"),
    )
    op, before, after = F.col("_op"), F.col("_before"), F.col("_after")

    def blank(c: Column) -> Column:
        # Chuỗi rỗng/toàn khoảng trắng: from_json trả NULL nhưng không đánh dấu hỏng -> chặn từ đây
        return F.trim(c) == ""

    rules = [   # (điều kiện, lý do, chi tiết) — thứ tự ưu tiên từ trên xuống
        (op.isNull() & before.isNull() & after.isNull(), "TOMBSTONE", None),
        (op == "t", "OP_TRUNCATE", None),
        (op == "m", "OP_MESSAGE", None),
        (op.isNull() | ~op.isin(*VALID_OPS), "INVALID_PAYLOAD",
         F.concat(F.lit("op không hợp lệ: "), F.coalesce(op, F.lit("NULL")))),
        (F.col("_src_object").isNull(), "INVALID_PAYLOAD", F.lit("thiếu source.table")),
        (~objects_cond(configured_objects), "UNCONFIGURED", None),
        (~objects_cond(scope_objects), "OUT_OF_SCOPE", None),
        (op.isin(*UPSERT_OPS) & after.isNull(), "INVALID_PAYLOAD", F.lit("thiếu after")),
        (op.isin(*UPSERT_OPS) & blank(after), "INVALID_PAYLOAD", F.lit("after rỗng")),
        ((op == "d") & before.isNull(), "INVALID_PAYLOAD", F.lit("thiếu before")),
        ((op == "d") & blank(before), "INVALID_PAYLOAD", F.lit("before rỗng")),
        (F.col("_cdc_ts_ms").isNull(), "MISSING_EVENT_ORDER", F.lit("thiếu source.ts_ms")),
        (F.col("_raw_ingested_at").isNull(), "MISSING_RAW_CURSOR", F.lit("thiếu thời điểm ingest")),
    ]
    reason, detail = F.lit(None).cast("string"), F.lit(None).cast("string")
    for cond, r, d in reversed(rules):
        reason = F.when(cond, F.lit(r)).otherwise(reason)
        detail = F.when(cond, d if d is not None else F.lit(None).cast("string")).otherwise(detail)

    df = (
        df.withColumn("_reason", reason)
          .withColumn("_reject_detail", detail)
          .withColumn("_status", F.when(F.col("_reason").isNull(), "VALID")
                                  .when(F.col("_reason").isin(*IGNORE_REASONS), "IGNORED")
                                  .otherwise("REJECTED"))
    )
    # Event bỏ qua không cần payload -> bỏ cho cache nhẹ
    for c in ("_before", "_after", "_source"):
        df = df.withColumn(c, F.when(F.col("_status") != "IGNORED", F.col(c)))
    return df


def profile_events(classified: DataFrame) -> Profile:
    """1 job: đếm theo trạng thái/lý do/bảng nguồn + max thời điểm ingest của mọi dòng đã đọc (kể cả IGNORED)."""
    rows = (
        classified.groupBy("_status", "_reason", "_src_object_schema", "_src_object")
        .agg(F.count(F.lit(1)).alias("n"), F.max("_raw_ingested_at").alias("max_ts"))
        .collect()
    )
    p = Profile()
    for r in rows:
        n = r["n"]
        p.read_rows += n
        if r["max_ts"] is not None and (p.watermark_to is None or r["max_ts"] > p.watermark_to):
            p.watermark_to = r["max_ts"]
        if r["_status"] == "VALID":
            p.valid_rows += n
            key = (r["_src_object_schema"], r["_src_object"])
            p.valid_by_src[key] = p.valid_by_src.get(key, 0) + n
        elif r["_status"] == "IGNORED":
            p.ignored_rows += n
            label = ".".join(x for x in (r["_src_object_schema"], r["_src_object"]) if x) or "(null)"
            bucket = p.ignored_detail.setdefault(r["_reason"], {})
            bucket[label] = bucket.get(label, 0) + n
        else:
            p.rejected_rows += n
    return p


def reject_rows(df: DataFrame, ctx: RunContext, reason: Column, detail: Column,
                trg_schema=None, trg_tbl=None) -> DataFrame:
    """Chuẩn hoá event lỗi theo schema ctrl_cdc_reject. event_hash = sha256(source, op, before, after)
    giúp không ghi trùng khi cùng event được đọc lại (overlap).
    trg_schema/trg_tbl: chuỗi (reject theo bảng) hoặc Column (reject phân loại, tra theo bảng nguồn — xem target_of)."""
    def as_col(v) -> Column:
        return v if isinstance(v, Column) else F.lit(v).cast("string")

    now = F.current_timestamp()
    payload = [F.coalesce(F.col(c), F.lit("")) for c in ("_source", "_op", "_before", "_after")]
    return df.select(
        F.sha2(F.concat_ws("\u0001", *payload), 256).alias("event_hash"),
        F.lit(ctx.pl_name).alias("pl_name"),
        F.lit(ctx.src_schema).alias("src_schema"),
        F.lit(ctx.src_tbl).alias("src_tbl"),
        F.col("_src_object").alias("src_object"),
        F.col("_src_db").alias("src_db"),
        as_col(trg_schema).alias("trg_schema"),
        as_col(trg_tbl).alias("trg_tbl"),
        F.col("_op").alias("cdc_op"),
        F.lit(None).cast("string").alias("entity_key"),
        F.col("_cdc_ts_ms").alias("cdc_ts_ms"),
        F.col("_cdc_lsn").alias("cdc_lsn"),
        F.col("_raw_ingested_at").alias("raw_ingested_at"),
        reason.alias("reject_reason"),
        detail.alias("reject_detail"),
        F.col("_before").alias("before_payload"),
        F.col("_after").alias("after_payload"),
        F.col("_source").alias("source_payload"),
        F.lit(ctx.run_id).alias("first_run_id"),
        F.lit(ctx.run_id).alias("last_run_id"),
        F.lit(1).alias("reject_count"),
        now.alias("first_rejected_at"),
        now.alias("last_rejected_at"),
        F.lit(False).alias("is_resolved"),
        F.lit(None).cast("timestamp").alias("resolved_at"),
    )

StatementMeta(, 58ce84c9-a336-4b3c-843e-6dba318d52af, 19, Finished, Available, Finished, False)

## B. Xử lý 1 bảng: parse → lọc event cũ theo state → gộp TOAST → MERGE

| Hàm | Mục đích |
|---|---|
| `event_order()`, `load_state(specs)` | **[03/10 → LIB]** Chuyển sang lib §13, logic không đổi |
| `build_table_events(spec, valid_df, state_df)` | 1 dòng/event: parse JSON, chuyển kiểu, cờ reject / cũ / áp dụng / NULL do chuyển kiểu / TOAST thay thế (`_unavailable__`) |
| `table_stats(spec, events)` | 1 job đếm mọi số liệu của bảng |
| `resolve_latest(spec, events)` | Gộp TOAST (giá trị gần nhất được cung cấp, kể cả NULL thật) + cờ `_has_value__`, rồi giữ 1 dòng mới nhất/entity |
| `build_merge_sql(spec, target, view)` | Câu MERGE upsert + xoá mềm; TOAST: `_has_value__` false (chỉ có giá trị thay thế) = giữ giá trị đích |
| `merge_into_target(spec, latest, ctx)` | Chạy MERGE qua `run_merge` của thư viện |
| `state_rows(spec, latest, ctx)` | **[03/10 → LIB]** Chuyển sang lib §13 |
| `_run_table(res, ...)` / `process_table(spec, ...)` | Các bước của 1 bảng; `cast_null_policy = FAIL` + có giá trị thành NULL khi chuyển kiểu → `CastNullError` trước MERGE. **[03/10 → LIB]** retry + log: `process_with_retry` (lib §14) |


In [5]:
# =============================================================================
# B. XỬ LÝ 1 BẢNG
# =============================================================================
# [03/10 → LIB] event_order(), load_state(specs) chuyển sang NB_LIB_EXTRACT_RAWDATA §13 (logic không đổi)


def build_table_events(spec: TableSpec, valid_df: DataFrame, state_df: Optional[DataFrame]) -> DataFrame:
    """Event của 1 bảng -> 1 dòng/event với cột đã chuyển kiểu và các cờ:
    _parse_failed (payload không phải JSON hợp lệ), _key_null (reject), _is_stale (cũ hơn hoặc bằng state),
    _applicable (được áp dụng), _cn__<cột> (giá trị có ở raw nhưng thành NULL do chuyển kiểu),
    _unavailable__<cột TOAST> (raw là giá trị thay thế __debezium_unavailable_value; luôn true/false, không NULL)."""
    payload = F.when(F.col("_op") == "d", F.col("_before")).otherwise(F.col("_after"))   # xoá: dữ liệu ở before
    top_paths = [c.json_path for c in spec.columns if "." not in c.json_path]

    def raw_value(c: ColumnSpec) -> Column:
        if "." in c.json_path:                                   # đường dẫn lồng nhau
            return F.get_json_object(payload, f"$.{c.json_path}")
        return F.col("_p").getField(c.json_path)

    def typed_value(c: ColumnSpec) -> Column:
        value = F.expr(convert_sql(c.raw_col, c.data_type, c.convert_rule))
        if c.is_toast and not c.is_key:
            # Giá trị TOAST thay thế = "không đổi" -> NULL, lấy lại giá trị thật ở bước gộp/MERGE
            value = F.when(F.col(c.raw_col) == F.lit(TOAST_SENTINEL), F.lit(None).cast(c.data_type)).otherwise(value)
        return value.alias(c.name)

    def cast_null_flag(c: ColumnSpec) -> Column:
        return (F.col(c.raw_col).isNotNull() & (F.col(c.raw_col) != F.lit(TOAST_SENTINEL))
                & F.col(c.name).isNull()).alias(f"_cn__{c.name}")

    def unavailable_flag(c: ColumnSpec) -> Column:
        # coalesce: raw NULL (NULL thật) phải là False, không được thành NULL -> bị hiểu nhầm là "không có giá trị"
        return F.coalesce(F.col(c.raw_col) == F.lit(TOAST_SENTINEL), F.lit(False)).alias(f"_unavailable__{c.name}")

    toast = set(spec.toast_cols)

    meta = ["_op", "_before", "_after", "_source", "_src_object", "_src_db",
            "_cdc_ts_ms", "_cdc_lsn", "_raw_ingested_at"]
    raw_cols = [c.raw_col for c in spec.columns]
    df = (
        valid_df.where((F.col("_src_object") == spec.src_object)
                       & (F.lit(spec.src_object_schema is None)
                          | (F.col("_src_object_schema") == F.lit(spec.src_object_schema))))
        .select(*meta, parse_json_strings(payload, top_paths).alias("_p"))
        .select(*meta, is_json_corrupt(F.col("_p"), payload).alias("_parse_failed"),
                *[raw_value(c).alias(c.raw_col) for c in spec.columns])
        .select(*meta, "_parse_failed", *raw_cols, *[typed_value(c) for c in spec.columns])
        .select(*meta, "_parse_failed", *[c.name for c in spec.columns],
                *[cast_null_flag(c) for c in spec.columns if c.can_cast_null],
                *[unavailable_flag(c) for c in spec.columns if c.name in toast])
    )

    keys = spec.key_cols
    key_null = F.col("_parse_failed") | reduce(
        lambda a, b: a | b, [F.col(k).isNull() | (F.trim(F.col(k).cast("string")) == "") for k in keys])
    # Khoá đơn: giá trị dạng chuỗi; khoá ghép: nối bằng "|" theo key_order (cùng định dạng state cũ)
    entity_key = (F.col(keys[0]).cast("string") if len(keys) == 1
                  else F.concat_ws("|", *[F.col(k).cast("string") for k in keys]))
    df = (
        df.withColumn("_key_null", key_null)
          .withColumn("_entity_key", F.when(~F.col("_key_null"), entity_key))
          .withColumn("_is_delete", F.when(F.col("_op") == "d", 1).otherwise(0))
    )

    if state_df is None:                                         # full_reload: không dùng state
        is_stale = F.lit(False)
    else:
        st = (state_df.where((F.col("trg_schema") == spec.trg_schema) & (F.col("trg_tbl") == spec.trg_tbl))
                      .select("_st_key", "_st_order"))
        df = df.join(st, F.col("_entity_key") == F.col("_st_key"), "left")
        is_stale = F.col("_st_key").isNotNull() & ~(event_order() > F.col("_st_order"))

    df = (
        df.withColumn("_is_stale", ~F.col("_key_null") & F.coalesce(is_stale, F.lit(False)))
          .withColumn("_applicable", ~F.col("_key_null") & ~F.col("_is_stale"))
    )
    # Payload gốc chỉ giữ cho dòng bị reject -> cache nhẹ
    for c in ("_before", "_after", "_source"):
        df = df.withColumn(c, F.when(F.col("_key_null"), F.col(c)))
    return df.drop("_st_key", "_st_order")


def table_stats(spec: TableSpec, events: DataFrame) -> dict:
    """1 job cho toàn bộ số đếm của bảng (input, reject, cũ, áp dụng, upsert, xoá, entity, NULL do chuyển kiểu/cột)."""
    applicable = F.col("_applicable")

    def as_int(cond: Column) -> Column:
        return F.sum(F.when(cond, 1).otherwise(0))

    aggs = [
        F.count(F.lit(1)).alias("input_rows"),
        as_int(F.col("_key_null")).alias("rejected_rows"),
        as_int(F.col("_is_stale")).alias("stale_rows"),
        as_int(applicable).alias("applicable_rows"),
        as_int(applicable & (F.col("_op") != "d")).alias("upsert_event_rows"),
        as_int(applicable & (F.col("_op") == "d")).alias("delete_event_rows"),
        F.countDistinct(F.when(applicable, F.col("_entity_key"))).alias("entity_rows"),
    ]
    aggs += [as_int(applicable & F.col(f"_cn__{c.name}")).alias(f"_cn__{c.name}")
             for c in spec.columns if c.can_cast_null]
    row = events.agg(*aggs).first().asDict()
    return {k: int(v or 0) for k, v in row.items()}


def resolve_latest(spec: TableSpec, events: DataFrame) -> DataFrame:
    """Event được áp dụng -> 1 dòng/entity.
    Cột TOAST: giá trị của event gần nhất CÓ cung cấp giá trị (không phải giá trị thay thế, không phải event xoá),
    kể cả NULL thật -> bọc struct để last(ignorenulls) không bỏ qua NULL thật. _has_value__<cột> = false khi mọi
    event trong batch chỉ có giá trị thay thế (MERGE giữ giá trị đích). Cột khác: theo event mới nhất.
    Event xoá bị loại khỏi bước điền: before của event xoá (REPLICA IDENTITY DEFAULT) có thể chỉ có cột khoá."""
    if spec.toast_cols:
        w_fill = (Window.partitionBy("_entity_key").orderBy(*ORDER_ASC)
                  .rowsBetween(Window.unboundedPreceding, Window.currentRow))
        known = {c: F.last(F.when(~F.col(f"_unavailable__{c}") & (F.col("_op") != "d"),
                                  F.struct(F.col(c).alias("value"))),
                           ignorenulls=True).over(w_fill)
                 for c in spec.toast_cols}
        events = events.select(*[F.col(x) for x in events.columns if x not in known],
                               *[k.alias(f"_known__{c}") for c, k in known.items()])
        events = events.select(*[F.col(x) for x in events.columns if not x.startswith("_known__")],
                               *[F.col(f"_known__{c}").getField("value").alias(c) for c in known],
                               *[F.col(f"_known__{c}").isNotNull().alias(f"_has_value__{c}") for c in known])
    w_last = Window.partitionBy("_entity_key").orderBy(*ORDER_DESC)
    return events.withColumn("_rn", F.row_number().over(w_last)).where("_rn = 1").drop("_rn")


def build_merge_sql(spec: TableSpec, target: str, view: str) -> str:
    """Câu MERGE: khớp + xoá -> xoá mềm; khớp -> cập nhật; không khớp + không phải xoá -> insert; không khớp + xoá -> bỏ qua.
    Cột TOAST: s._has_value__<cột> = true -> lấy nguồn (kể cả NULL thật); false (chỉ có giá trị thay thế) -> giữ đích.
    Insert entity mới chỉ có giá trị thay thế -> NULL (Debezium không gửi giá trị; cần snapshot lại)."""
    toast = set(spec.toast_cols)
    on = " AND ".join(f"t.`{k}` = s.`{k}`" for k in spec.key_cols)
    updates = [
        (f"t.`{c.name}` = CASE WHEN s.`_has_value__{c.name}` THEN s.`{c.name}` ELSE t.`{c.name}` END"
         if c.name in toast else f"t.`{c.name}` = s.`{c.name}`")
        for c in spec.columns if not c.is_key
    ]
    updates += ["t.`deleted` = false", "t.`_ingested_at` = current_timestamp()", "t.`_source_db` = s.`_src_db`"]
    insert_cols = [f"`{c.name}`" for c in spec.columns] + ["`deleted`", "`_ingested_at`", "`_source_db`"]
    insert_vals = [f"s.`{c.name}`" for c in spec.columns] + ["false", "current_timestamp()", "s.`_src_db`"]
    sep = ",\n        "
    return (
        f"MERGE INTO {target} AS t\n"
        f"USING {view} AS s\n"
        f"ON {on}\n"
        f"WHEN MATCHED AND s.`_cdc_op` = 'd' THEN UPDATE SET\n"
        f"        t.`deleted` = true,\n        t.`_ingested_at` = current_timestamp()\n"
        f"WHEN MATCHED THEN UPDATE SET\n        {sep.join(updates)}\n"
        f"WHEN NOT MATCHED AND s.`_cdc_op` <> 'd' THEN INSERT\n"
        f"        ({', '.join(insert_cols)})\n"
        f"        VALUES ({', '.join(insert_vals)})"
    )


def merge_into_target(spec: TableSpec, latest: DataFrame, ctx: RunContext) -> tuple:
    """MERGE dòng mới nhất mỗi entity vào bảng đích -> (inserted, updated). Temp view tên riêng theo exec_id + bảng."""
    source = latest.select(*[c.name for c in spec.columns], *[f"_has_value__{c}" for c in spec.toast_cols],
                           F.col("_op").alias("_cdc_op"), "_src_db")
    return run_merge(source, spec.fq_name, lambda target, view: build_merge_sql(spec, target, view),
                     f"_v_{ctx.exec_id[:8]}_{spec.trg_tbl}")


# [03/10 → LIB] state_rows(spec, latest, ctx) chuyển sang NB_LIB_EXTRACT_RAWDATA §13
#               (khác duy nhất: is_deleted = coalesce(_op = 'd', false); event partner luôn có op nên kết quả như cũ)


def _run_table(res: TableResult, valid_df: DataFrame, state_df: Optional[DataFrame], ctx: RunContext) -> None:
    """Các bước của 1 bảng; ghi kết quả vào res. Lỗi -> raise cho process_table xử lý."""
    spec = res.spec
    note = ensure_target(spec, ctx.dry_run)
    if note:
        log(note, wave=res.wave, table=spec.trg_tbl)

    events = build_table_events(spec, valid_df, state_df).persist(StorageLevel.MEMORY_AND_DISK)
    res.cached.append(events)
    stats = table_stats(spec, events)
    for k in ("input_rows", "rejected_rows", "stale_rows", "applicable_rows",
              "entity_rows", "upsert_event_rows", "delete_event_rows"):
        setattr(res, k, stats[k])
    res.cast_null_detail = {k[len("_cn__"):]: v for k, v in stats.items() if k.startswith("_cn__") and v}
    res.cast_null_rows = sum(res.cast_null_detail.values())
    if res.cast_null_detail:
        if ctx.cast_null_policy == "FAIL":
            # Trước MERGE: không ghi đích, không ghi state -> bảng FAILED, watermark giữ, lần sau đọc lại đủ event
            raise CastNullError(
                f"{res.cast_null_rows} giá trị thành NULL khi chuyển kiểu {res.cast_null_detail}. Sửa data_type/"
                f"convert_rule trong ctrl_cfg_schema_registry rồi chạy lại bình thường; hoặc chấp nhận NULL: chạy "
                f"cast_null_policy = WARN")
        log(f"giá trị thành NULL khi chuyển kiểu: {res.cast_null_detail}", "WARN", res.wave, spec.trg_tbl)

    if res.rejected_rows:
        missing_keys = F.concat_ws(",", *[F.when(F.col(k).isNull() | (F.trim(F.col(k).cast("string")) == ""), F.lit(k))
                                          for k in spec.key_cols])
        parse_failed = F.col("_parse_failed")
        res.reject_df = reject_rows(
            events.where("_key_null"), ctx,
            F.when(parse_failed, F.lit("INVALID_PAYLOAD")).otherwise(F.lit("MISSING_ENTITY_KEY")),
            F.when(parse_failed, F.lit("before/after không phải JSON hợp lệ"))
             .otherwise(F.concat(F.lit("khoá NULL/rỗng hoặc không chuyển được kiểu: "), missing_keys)),
            spec.trg_schema, spec.trg_tbl,
        )
    if res.applicable_rows == 0 or ctx.dry_run:
        return

    latest = resolve_latest(spec, events.where("_applicable")).persist(StorageLevel.MEMORY_AND_DISK)
    res.cached.append(latest)
    res.inserted_rows, res.updated_rows = merge_into_target(spec, latest, ctx)
    res.state_df = state_rows(spec, latest, ctx)


def process_table(spec: TableSpec, valid_df: DataFrame, state_df: Optional[DataFrame],
                  ctx: RunContext, wave: int) -> TableResult:
    """Chạy trong luồng của wave. Lỗi tạm thời -> thử lại (max_retries); lỗi khác -> FAILED.
    Không bao giờ raise: bảng lỗi không làm dừng bảng khác.
    [03/10 → LIB] thân hàm (retry, log, setJobDescription) chuyển sang process_with_retry của lib §14, logic không đổi."""
    return process_with_retry(spec, wave, ctx, lambda res: _run_table(res, valid_df, state_df, ctx))

StatementMeta(, 58ce84c9-a336-4b3c-843e-6dba318d52af, 20, Finished, Available, Finished, False)

## C. Chạy theo wave và commit

| Hàm | Mục đích |
|---|---|
| `run_waves(specs, profile, valid_df, state_df, ctx, results)` | Bảng 0 event → NO_DATA; bảng còn lại chạy theo priority. **[03/10 → LIB]** gọi `run_tables` (lib §14) |
| `commit_state(results)` | **[03/10 → LIB]** Chuyển sang lib §13 (cùng `state_order_sql`, `STATE_NOT_OLDER`), logic không đổi |
| `target_of(specs, field)` | Tra `trg_schema`/`trg_tbl` theo bảng nguồn của event, để reject phân loại vẫn biết thuộc bảng đích nào |
| `commit_rejects(classified, profile, results, ctx)` | Gom event lỗi phân loại + lỗi theo bảng. **[03/10 → LIB]** MERGE theo `event_hash`: `merge_rejects` (lib §13) |
| `commit(...)` | Thứ tự commit: state → reject → table log (chỉ luồng chính ghi bảng control) |

In [6]:
# =============================================================================
# C. CHẠY THEO WAVE VÀ COMMIT
# =============================================================================
def run_waves(specs: list, profile: Profile, valid_df: DataFrame, state_df: Optional[DataFrame],
              ctx: RunContext, results: list) -> None:
    """Bảng không có event -> NO_DATA (không tốn job). Bảng còn lại chạy theo wave priority.
    [03/10 → LIB] thân hàm chuyển sang run_tables của lib §14 (NO_DATA, wave, stop_on_failure), logic không đổi.
    [03/10 → LIB] state_order_sql, STATE_NOT_OLDER, commit_state chuyển sang lib §13."""
    run_tables(specs, profile.events_for,
               lambda spec, wave: process_table(spec, valid_df, state_df, ctx, wave), ctx, results)


def target_of(specs: list, field: str) -> Column:
    """Bảng đích của event theo bảng nguồn (_src_object_schema, _src_object). Không khớp spec nào -> NULL.
    Input : specs — list[TableSpec] ; field — "trg_schema" | "trg_tbl"
    Output: Column chuỗi. Vd event products/public -> "slv_products" ; event thiếu source.table -> NULL"""
    expr = F.lit(None).cast("string")
    for s in reversed(specs):                                    # spec đứng trước được ưu tiên
        cond = F.col("_src_object") == F.lit(s.src_object)
        if s.src_object_schema is not None:
            cond = cond & (F.col("_src_object_schema") == F.lit(s.src_object_schema))
        expr = F.when(cond, F.lit(getattr(s, field))).otherwise(expr)
    return expr


def commit_rejects(classified: DataFrame, profile: Profile, results: list, ctx: RunContext) -> None:
    """Event lỗi phân loại + thiếu khoá của bảng SUCCESS -> MERGE theo event_hash.
    Event đã có (đọc lại do overlap) -> reject_count + 1, không sinh dòng trùng."""
    parts = [r.reject_df for r in results if r.status == "SUCCESS" and r.reject_df is not None]
    if profile.rejected_rows:
        specs = [r.spec for r in results]                        # mọi bảng trong phạm vi đều có TableResult
        parts.append(reject_rows(classified.where("_status = 'REJECTED'"), ctx,
                                 F.col("_reason"), F.col("_reject_detail"),
                                 target_of(specs, "trg_schema"), target_of(specs, "trg_tbl")))
    # [03/10 → LIB] MERGE theo event_hash chuyển sang merge_rejects của lib §13 (cùng điều kiện, cùng cột cập nhật)
    # [03/10 → LIB] truyền ctx.src_tbl: MERGE chỉ đọc partition partner_raw_data của ctrl_cdc_reject (partition src_tbl,
    #               NB_CREATE_DDL 03/10 - 3P) -> không xung đột ghi với luồng 3rd-party chạy song song
    merge_rejects(parts, f"event lỗi phân loại={profile.rejected_rows}, "
                         f"lỗi theo bảng={sum(r.rejected_rows for r in results if r.status == 'SUCCESS')}", ctx.src_tbl)


def commit(classified: DataFrame, profile: Profile, results: list, ctx: RunContext) -> None:
    """Thứ tự: state -> reject -> table log. Chết giữa chừng: lần sau đọc lại, state lọc event đã áp dụng."""
    commit_state(results, ctx.src_tbl)         # [03/10 → LIB] + src_tbl: MERGE chỉ đọc partition của partner (xem commit_rejects)
    commit_rejects(classified, profile, results, ctx)
    append_rows(T_LOG_TABLE_RUN, [table_log_row(r, ctx) for r in results])

StatementMeta(, 58ce84c9-a336-4b3c-843e-6dba318d52af, 21, Finished, Available, Finished, False)

## Main

`build_context()` đọc tham số thành `RunContext`; `run_extract()` là luồng chính, trả JSON tóm tắt cho cell exit.
**[03/10 → LIB]** `run_extract()` = nạp cấu hình + watermark rồi gọi `run_controlled` (lib §14: khoá, run log, watermark, nhả khoá);
phần riêng của partner nằm ở `process_run()` (đọc raw → phân loại → wave → commit).

In [7]:
# =============================================================================
# MAIN
# =============================================================================
def build_context() -> RunContext:
    """Tham số notebook -> RunContext. run_mode: DRY_RUN > FULL_RELOAD > INCREMENTAL (nhãn ghi log)."""
    if not is_fq_name(f"{src_schema}.{src_tbl}"):
        raise ConfigError(f"src_schema/src_tbl không hợp lệ: {src_schema}.{src_tbl}")
    policy = str(cast_null_policy or "").strip().upper()
    if policy not in CAST_NULL_POLICIES:
        raise ConfigError(f"cast_null_policy phải là {' | '.join(CAST_NULL_POLICIES)}, đang là {cast_null_policy!r}")
    exec_id = str(uuid.uuid4())
    filters = split_csv(table_filter)
    params = {
        "pl_name": pl_name, "run_id": run_id, "src_schema": src_schema, "src_tbl": src_tbl,
        "overlap_minutes": int(overlap_minutes), "max_parallel": int(max_parallel), "max_retries": int(max_retries),
        "table_filter": filters, "full_reload": to_bool(full_reload), "dry_run": to_bool(dry_run),
        "stop_on_failure": to_bool(stop_on_failure), "running_timeout_minutes": int(running_timeout_minutes),
        "allow_full_scan": to_bool(allow_full_scan), "cast_null_policy": policy,
    }
    return RunContext(
        exec_id=exec_id, run_id=str(run_id or "").strip() or exec_id, pl_name=pl_name, nb_name=NB_NAME,
        src_schema=src_schema, src_tbl=src_tbl,
        run_mode="DRY_RUN" if params["dry_run"] else "FULL_RELOAD" if params["full_reload"] else "INCREMENTAL",
        table_filter=filters, overlap_minutes=params["overlap_minutes"], max_parallel=params["max_parallel"],
        max_retries=params["max_retries"], full_reload=params["full_reload"], dry_run=params["dry_run"],
        stop_on_failure=params["stop_on_failure"], running_timeout_minutes=params["running_timeout_minutes"],
        run_params=json.dumps(params, ensure_ascii=False), started_at=utc_now(),
        allow_full_scan=params["allow_full_scan"], cast_null_policy=policy,
    )


def process_run(ctx: RunContext, specs: list, configured: set, out: RunOutcome) -> None:
    """Phần riêng của partner trong 1 lần chạy (chạy bên trong run_controlled, sau khi đã nhận khoá + chốt version):
    đọc raw -> phân loại -> profile -> state -> wave -> trạng thái -> commit. Điền out để run_controlled ghi log/watermark.
    [03/10 → LIB] tách từ run_extract cũ; các bước và thứ tự giữ nguyên."""
    t0 = time.time()
    classified = classify_events(read_raw(ctx), [s.src_ref for s in specs], configured) \
        .persist(StorageLevel.MEMORY_AND_DISK)
    out.cached.append(classified)
    log(f"đọc raw: {ctx.read_mode} | {ctx.read_note or ''}")
    out.profile = profile_events(classified)
    log(f"đọc {out.profile.read_rows} | hợp lệ {out.profile.valid_rows} | bỏ qua {out.profile.ignored_rows} "
        f"| lỗi {out.profile.rejected_rows} | watermark_to {out.profile.watermark_to} ({time.time() - t0:.0f}s)")

    state_df = None
    if not ctx.full_reload:
        state_df = load_state(specs).persist(StorageLevel.MEMORY_AND_DISK)
        out.cached.append(state_df)
    run_waves(specs, out.profile, classified.where("_status = 'VALID'"), state_df, ctx, out.results)
    out.status = decide_status(out.results)
    if not ctx.dry_run:
        t0 = time.time()
        commit(classified, out.profile, out.results, ctx)
        log(f"commit xong ({time.time() - t0:.0f}s)")


def run_extract() -> dict:
    """Luồng chính. Cấu hình sai -> raise trước khi ghi gì. Không nhận được khoá -> SKIPPED_CONCURRENT.
    Lỗi khác -> FAILED, vẫn đóng watermark, run log và nhả khoá.
    [03/10 → LIB] khoá / run log RUNNING / chốt version / watermark / đóng run log / nhả khoá / giải phóng cache
    chuyển sang run_controlled của lib §14 (thứ tự và xử lý lỗi giữ nguyên); phần riêng ở process_run."""
    ctx = build_context()
    specs, configured = load_table_specs(ctx)          # lib mặc định allowed_modes=("CDC",)
    load_watermark(ctx)
    log(f"{ctx.raw_fq} | mode={ctx.run_mode} | {len(specs)} bảng | last_src_version={ctx.src_version_from} "
        f"| watermark={ctx.watermark_from} | exec_id={ctx.exec_id}")
    return run_controlled(ctx, lambda out: process_run(ctx, specs, configured, out))


RUN_SUMMARY = run_extract()

StatementMeta(, 58ce84c9-a336-4b3c-843e-6dba318d52af, 22, Finished, Available, Finished, False)

[09:37:14][INFO] lh_vv_bronze.dbo.partner_raw_data | mode=INCREMENTAL | 23 bảng | last_src_version=None | watermark=None | exec_id=de379b89-bb6d-493e-b236-6ab8462ed7bc
[09:37:37][INFO] nhận khoá (WATERMARK)
[09:38:04][INFO] mở run log, chốt version 18543 (28s)
[09:38:10][INFO] đọc raw: FULL | chưa có last_src_version (lần đầu / sau reset / sau cutover) | allow_full_scan
[09:39:54][INFO] đọc 1517296 | hợp lệ 1285720 | bỏ qua 32158 | lỗi 199418 | watermark_to 2026-09-17 10:22:45.180708 (110s)
[09:39:57][INFO][w1] priority 1: 10 việc ['slv_pn_business_service_i18ns', 'slv_pn_business_service_poi_link', 'slv_pn_business_services', 'slv_pn_hotel_rate_plans', 'slv_pn_hotel_room_types', 'slv_pn_order_item_hotels', 'slv_pn_order_item_tickets', 'slv_pn_order_refs', 'slv_pn_orders', 'slv_pn_partners']
[09:40:01][INFO][w1][slv_pn_business_service_i18ns] tạo bảng mới
[09:40:01][INFO][w1][slv_pn_business_services] tạo bảng mới
[09:40:02][INFO][w1][slv_pn_business_service_poi_link] tạo bảng mới
[09:

In [8]:
# =============================================================================
# EXIT (ngoài try/except; pipeline đọc exitValue)
# =============================================================================
_summary_json = json.dumps(RUN_SUMMARY, ensure_ascii=False, default=str)
print(json.dumps(RUN_SUMMARY, ensure_ascii=False, indent=2, default=str))
# SKIPPED_CONCURRENT cũng raise: pipeline đã đặt Concurrency = 1 nên gặp trường hợp này là có run treo/chạy tay chồng
# (error trong exit JSON ghi exec_id đang giữ khoá)
if RUN_SUMMARY["status"] in ("FAILED", "PARTIAL_FAILED", "SKIPPED_CONCURRENT"):
    raise RuntimeError(f"{NB_NAME} {RUN_SUMMARY['status']}: {_summary_json}")
notebookutils.notebook.exit(_summary_json)

StatementMeta(, 58ce84c9-a336-4b3c-843e-6dba318d52af, 23, Finished, Available, Finished, False)

{
  "status": "SUCCESS",
  "exec_id": "de379b89-bb6d-493e-b236-6ab8462ed7bc",
  "run_id": "de379b89-bb6d-493e-b236-6ab8462ed7bc",
  "run_mode": "INCREMENTAL",
  "src": "lh_vv_bronze.dbo.partner_raw_data",
  "src_version_from": null,
  "src_version": 18543,
  "read_mode": "FULL",
  "read_note": "chưa có last_src_version (lần đầu / sau reset / sau cutover) | allow_full_scan",
  "watermark_from": null,
  "watermark_to": "2026-09-17 10:22:45.180708",
  "tables": {
    "total": 23,
    "ok": 10,
    "failed": 0,
    "skipped": 0,
    "no_data": 13
  },
  "failed_tables": [],
  "rows": {
    "read": 1517296,
    "valid": 1285720,
    "ignored": 32158,
    "rejected": 199418,
    "inserted": 815,
    "updated": 0,
    "cast_null": 0
  },
  "duration_s": 312.8,
  "error": null
}
ExitValue: {"status": "SUCCESS", "exec_id": "de379b89-bb6d-493e-b236-6ab8462ed7bc", "run_id": "de379b89-bb6d-493e-b236-6ab8462ed7bc", "run_mode": "INCREMENTAL", "src": "lh_vv_bronze.dbo.partner_raw_data", "src_version_

## Vận hành

**Notebook activity** — FL_00 không có Switch. Tham số gắn cứng `pl_name` = `PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00`, `src_tbl` = `partner_raw_data`, `run_id` = rỗng. Đặt Concurrency = 1 cho pipeline; `running_timeout_minutes` = 780 (13 giờ). Timeout activity là 59 phút, ngắn hơn hạn khoá.
Cần cột khoá: `ALTER TABLE lh_vv_bronze.ctrl.ctrl_mng_watermark ADD COLUMNS (lock_exec_id STRING, lock_at TIMESTAMP)` — chưa có thì notebook WARN và chống chạy chồng bằng run log (không nguyên tử).
Notebook `NB_LIB_EXTRACT_RAWDATA` phải nằm cùng workspace và cùng default lakehouse (`lh_vv_bronze`).

| Việc | Tham số |
|---|---|
| Chạy thử, không ghi gì | `dry_run = True` |
| Nạp lại 1 vài bảng từ toàn bộ raw | `table_filter = "slv_products,slv_orders"`, `full_reload = True` (watermark không đổi; tạm dừng lịch pipeline khi chạy lâu) |
| Chạy lại sau lỗi | Chạy lại bình thường: state bỏ event đã áp dụng, bảng lỗi được xử lý lại |
| Bảng FAILED vì `CastNullError` (giá trị thành NULL khi chuyển kiểu) | Sửa `data_type`/`convert_rule` trong `ctrl_cfg_schema_registry` rồi chạy lại bình thường; hoặc chấp nhận NULL: chạy 1 lần `cast_null_policy = "WARN"` |
| `SKIPPED_CONCURRENT` mà không có run nào đang chạy (run trước bị kill, khoá chưa hết hạn) | Chờ quá `running_timeout_minutes`, hoặc nhả tay: `UPDATE lh_vv_bronze.ctrl.ctrl_mng_watermark SET lock_exec_id = NULL, lock_at = NULL WHERE watermark_id = '<id>' AND lock_exec_id = '<exec_id trong lỗi>'` |
| Lần đầu / sau reset / sau cutover, hoặc run FAILED vì `VersionGapError` | Chạy 1 lần với `allow_full_scan = True`: đọc toàn bộ raw, state lọc event đã áp dụng, tiến watermark khi thành công. Các lần sau để `False` |

**Kiểm tra sau mỗi lần chạy**
```sql
SELECT status, run_mode, read_rows, valid_rows, ignored_rows, rejected_rows, tbl_success_count, tbl_failed_count,
       watermark_from, watermark_to, duration_ms, error_message
FROM lh_vv_bronze.ctrl.ctrl_log_run ORDER BY started_at DESC LIMIT 10;

SELECT trg_tbl, status, attempt_no, input_rows, stale_rows, applicable_rows, inserted_rows, updated_rows,
       delete_event_rows, cast_null_rows, rejected_rows, duration_ms, error_message
FROM lh_vv_bronze.ctrl.ctrl_log_table_run WHERE exec_id = '<exec_id>' ORDER BY priority, trg_tbl;

SELECT trg_tbl, entity_key, COUNT(*) FROM lh_vv_bronze.ctrl.ctrl_cdc_state
GROUP BY trg_tbl, entity_key HAVING COUNT(*) > 1;   -- kỳ vọng 0 dòng
```

**Cách đọc raw** (`ctrl_log_run.read_mode`, `read_note`):

| read_mode | Khi nào | Ghi chú |
|---|---|---|
| `VERSION` | Bình thường (đã có `last_src_version`, table id raw không đổi) | Chỉ đọc file của commit mới; không phụ thuộc stats, không bỏ sót event commit trễ |
| `FULL` | `full_reload`, hoặc có khoảng trống và `allow_full_scan = True` | Quét cả raw; khi không `full_reload` thì state vẫn lọc event đã áp dụng nên không áp dụng lại |

Khoảng trống (không xác định được đủ commit cần đọc) → `allow_full_scan = False` thì run FAILED (`VersionGapError`), watermark giữ nguyên:
chưa có `last_src_version`; raw bị tạo lại (`last_src_table_id` đổi hoặc version lùi); thiếu file `_delta_log` (log > 30 ngày đã bị dọn);
file dữ liệu đã bị VACUUM; raw có DELETE/UPDATE/MERGE/RESTORE; raw bật deletion vector / column mapping / partition.
Không có kiểu lọc theo `EventProcessedUtcTime` (TIME, bỏ 02/10): có thể bỏ sót event commit trễ rồi vẫn tiến watermark qua chúng.
Không VACUUM raw với retention < 7 ngày; không giảm `delta.logRetentionDuration` của raw xuống dưới thời gian pipeline có thể dừng.

**Chống chạy chồng và chống lùi** (03/10):

| Lớp | Cách làm | Khi nào cần |
|---|---|---|
| Khoá | `UPDATE ctrl_mng_watermark SET lock_exec_id = <exec_id>, lock_at = now WHERE watermark_id = … AND (lock_exec_id IS NULL OR quá hạn)` rồi đọc lại; Delta optimistic concurrency cho đúng 1 run nhận khoá. Nhả trong `finally` | Chặn chính: 2 run (lịch + chạy tay, retry activity) khởi động cùng lúc |
| State không lùi | MERGE `ctrl_cdc_state` chỉ cập nhật dòng khớp khi `(ts_ms, lsn, ingest, del)` mới ≥ đang lưu | Run bị lấy khoá (chạy quá hạn) vẫn commit sau run mới |
| Watermark không lùi | Ghi SUCCESS chỉ khi còn giữ khoá và `last_src_version` NULL / ≤ version chốt / raw bị tạo lại; không ghi được → WARN, ghi vào `error_message` | Như trên |

Bảng đích không có cột thứ tự event nên MERGE đích không tự chặn lùi được — khoá là lớp bảo vệ chính; đặt `running_timeout_minutes` đủ lớn.
